# ASSIGNMENT 1 — MÔN HỌC MÁY & HÌNH HỌC TUYẾN TÍNH (MAI391)
**Tập dữ liệu:** Iris Plants Database (UCI / sklearn.datasets)
**Phạm vi:** Chương 3 (Analytic Geometry), Chương 4 (Matrix Decomposition), Chương 5 (Vector Calculus)

In [ ]:
# Import các thư viện cần thiết
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from pathlib import Path
from sklearn.datasets import load_iris

# Cấu hình hiển thị
np.random.seed(42)
torch.manual_seed(42)
np.set_printoptions(precision=6, suppress=True, linewidth=120)
pd.set_option('display.precision', 6)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

--- 
# NHIỆM VỤ A — CHƯƠNG 3: KHẢNG CÁCH, GÓC VÀ PHÉP CHIẾU

### A1. Nạp dữ liệu & Chuẩn hoá dữ liệu thô về mean=0, std=1
- **Nguồn dữ liệu:** Iris Dataset (150 mẫu, 4 đặc trưng số: sepal_length, sepal_width, petal_length, petal_width).
- **Công thức chuẩn hoá:** $x_{ij}^{(norm)} = \frac{x_{ij} - \mu_j}{\sigma_j}$ với ddof=1 (Sample standard deviation).

In [ ]:
# === NHIỆM VỤ A1 ===
# 1. Nạp dữ liệu Iris thô
iris = load_iris(as_frame=True)
cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
df_raw = iris.frame.copy()
df_raw.columns = cols + ["species"]

D_raw = df_raw[cols].to_numpy(dtype=float)
N, d = D_raw.shape
print(f"[A1] Kích thước ma trận dữ liệu thô D: {D_raw.shape} (N={N} mẫu, d={d} đặc trưng)")

# 2. Chuẩn hoá dữ liệu về trung bình 0 và độ lệch chuẩn 1 (Sample std, ddof=1)
mu = D_raw.mean(axis=0)
sd = D_raw.std(axis=0, ddof=1)
X = (D_raw - mu) / sd

print("[A1] Trung bình mu trước chuẩn hoá:", np.round(mu, 4))
print("[A1] Độ lệch chuẩn std trước chuẩn hoá:", np.round(sd, 4))
print("[A1] Trung bình ma trận X sau chuẩn hoá:", np.round(X.mean(axis=0), 10))
print("[A1] Độ lệch chuẩn ma trận X sau chuẩn hoá:", X.std(axis=0, ddof=1))
df_raw.head()

### A2. Tích vô hướng chuẩn $\langle x,y\rangle = x^\top y$
Chọn 5 mẫu bất kỳ và lập bảng $5\times5$ khoảng cách Euclid và $cos\,\theta$.

In [ ]:
# === NHIỆM VỤ A2 ===
rng = np.random.default_rng(1)
idx = np.sort(rng.choice(N, size=5, replace=False))
S_samples = X[idx]
labels = [f's{i}' for i in idx]

def dist_matrix(S, A=None):
    A = np.eye(S.shape[1]) if A is None else A
    n_s = len(S)
    M = np.zeros((n_s, n_s))
    for i in range(n_s):
        for j in range(n_s):
            diff = S[i] - S[j]
            M[i, j] = np.sqrt(diff @ A @ diff)
    return M

def cos_matrix(S, A=None):
    A = np.eye(S.shape[1]) if A is None else A
    n_s = len(S)
    M = np.zeros((n_s, n_s))
    for i in range(n_s):
        for j in range(n_s):
            M[i, j] = (S[i] @ A @ S[j]) / (np.sqrt(S[i] @ A @ S[i]) * np.sqrt(S[j] @ A @ S[j]))
    return M

D_euc = pd.DataFrame(dist_matrix(S_samples), index=labels, columns=labels)
C_euc = pd.DataFrame(cos_matrix(S_samples), index=labels, columns=labels)

print("=== A2: Bảng khoảng cách Euclid d(xi, xj) ===")
display(D_euc)
print("=== A2: Bảng cos(theta_ij) ===")
display(C_euc)

### A3. Tích vô hướng có trọng số $\langle x,y\rangle_A = x^\top A y$
Chọn ma trận đường chéo dương $A = \text{diag}(1, 1, 3, 3)$ (gán trọng số lớn hơn 3 cho 2 đặc trưng petal).

In [ ]:
# === NHIỆM VỤ A3 ===
A_mat = np.diag([1, 1, 3, 3])
print("Kiểm tra A đối xứng:", np.allclose(A_mat, A_mat.T))
print("Giá trị riêng của A:", np.linalg.eigvalsh(A_mat))
np.linalg.cholesky(A_mat)
print("Cholesky thành công -> A là ma trận xác định dương.\n")

D_A = pd.DataFrame(dist_matrix(S_samples, A_mat), index=labels, columns=labels)
C_A = pd.DataFrame(cos_matrix(S_samples, A_mat), index=labels, columns=labels)

print("=== A3: Bảng khoảng cách có trọng số d_A(xi, xj) ===")
display(D_A)
print("=== A3: Bảng cos_A(theta_ij) có trọng số ===")
display(C_A)

### A4. So sánh thứ tự mẫu gần nhau nhất giữa hai tích vô hướng

In [ ]:
# === NHIỆM VỤ A4 ===
def get_neighbour_orders(M):
    out = {}
    for i, l in enumerate(labels):
        order = [labels[j] for j in np.argsort(M[i]) if j != i]
        out[l] = order
    return out

o_euc = get_neighbour_orders(D_euc.to_numpy())
o_A   = get_neighbour_orders(D_A.to_numpy())

cmp_rows = []
for l in labels:
    cmp_rows.append({
        'Mẫu': l,
        'Thứ tự (Euclid)': ' < '.join(o_euc[l]),
        'Thứ tự (A-weighted)': ' < '.join(o_A[l]),
        'Mẫu gần nhất giống?': o_euc[l][0] == o_A[l][0],
        'Toàn bộ thứ tự giống?': o_euc[l] == o_A[l]
    })
df_cmp = pd.DataFrame(cmp_rows)
display(df_cmp)

### A5. Thuật toán Gram-Schmidt và Phép chiếu trực giao
Lấy 2 đặc trưng đầu của 5 mẫu đầu tiên làm cơ sở $B=\{b_1, b_2\}$, chiếu mẫu $x$ (petal_length) lên không gian con $U = \text{span}\{b_1, b_2\}$.

In [ ]:
# === NHIỆM VỤ A5 ===
B_mat = D_raw[:5, :2]
b1, b2 = B_mat[:, 0], B_mat[:, 1]
x_vec = D_raw[:5, 2]

# Thuật toán Gram-Schmidt từng bước
u1 = b1 / np.linalg.norm(b1)
v2 = b2 - (b2 @ u1) * u1
u2 = v2 / np.linalg.norm(v2)

proj_x = (x_vec @ u1) * u1 + (x_vec @ u2) * u2
err_proj = np.linalg.norm(x_vec - proj_x)

print("=== A5: Kiểm chứng Gram-Schmidt & Phép chiếu ===")
print("u1 =", np.round(u1, 4))
print("u2 =", np.round(u2, 4))
print("Phép chiếu pi_U(x) =", np.round(proj_x, 4))
print("Vector sai số (x - pi_U(x)) =", np.round(x_vec - proj_x, 4))
print("Độ dài sai số ||x - pi_U(x)|| =", err_proj)

# Đối chiếu phương trình chuẩn (Normal equations)
proj_normal = B_mat @ np.linalg.solve(B_mat.T @ B_mat, B_mat.T @ x_vec)
print("Sai lệch tối đa so với Normal Equations:", np.abs(proj_x - proj_normal).max())

--- 
# NHIỆM VỤ B — CHƯƠNG 4: PHÂN TÍCH MA TRẬN (MATRIX DECOMPOSITION)

### B1. Ma trận hiệp phương sai $S$

In [ ]:
# === NHIỆM VỤ B1 ===
S_cov = (X.T @ X) / (N - 1)
print(f"Kích thước ma trận S: {S_cov.shape}")
print("Ma trận hiệp phương sai S:")
df_S = pd.DataFrame(S_cov, index=cols, columns=cols)
display(df_S)
print("Kiểm tra tính đối xứng max|S - S^T|:", np.abs(S_cov - S_cov.T).max())

### B2 & B3. Ma trận con $3\times3$ ($S_3$) và kiểm chứng bằng `numpy.linalg.eig`

In [ ]:
# === NHIỆM VỤ B2 & B3 ===
S3 = S_cov[:3, :3]

# Tính toán tự động chuẩn bằng Numpy đối chiếu lời giải làm tay
w_np, V_np = np.linalg.eig(S3)
order_idx = np.argsort(w_np)[::-1]
w_sorted = w_np[order_idx]
V_sorted = V_np[:, order_idx]

print("=== B2/B3: Kết quả từ numpy.linalg.eig ===")
print("det(S3) =", np.linalg.det(S3))
print("tr(S3)  =", np.trace(S3))
print("Các giá trị riêng (sắp xếp giảm dần):", w_sorted)
print("Vector riêng ứng với giá trị riêng lớn nhất lambda_1:", V_sorted[:, 0])

### B4. Xác minh hệ thức $\det S = \prod \lambda_i$ và $\operatorname{tr} S = \sum \lambda_i$

In [ ]:
# === NHIỆM VỤ B4 ===
eigvals_S = np.linalg.eigvalsh(S_cov)
det_direct = np.linalg.det(S_cov)
det_prod   = np.prod(eigvals_S)
tr_direct  = np.trace(S_cov)
tr_sum     = np.sum(eigvals_S)

print("=== B4: Xác minh hệ thức trên ma trận S (4x4) ===")
print(f"det(S) trực tiếp = {det_direct:.8f} | Tích các giá trị riêng = {det_prod:.8f}")
print(f"tr(S) trực tiếp  = {tr_direct:.8f} | Tổng các giá trị riêng  = {tr_sum:.8f}")
print("Sai lệch det:", abs(det_direct - det_prod))
print("Sai lệch tr :", abs(tr_direct - tr_sum))

### B5. Chéo hoá $S_3 = PDP^{-1}$, tính $S_3^4$ và so sánh thời gian

In [ ]:
# === NHIỆM VỤ B5 ===
import timeit

w3, P3 = np.linalg.eigh(S3)
D3 = np.diag(w3)
P3_inv = P3.T  # Do P3 trực giao

S3_4_direct = S3 @ S3 @ S3 @ S3
S3_4_diag   = P3 @ np.diag(w3**4) @ P3_inv

print("Sai lệch tối đa giữa 2 cách tính S3^4:", np.abs(S3_4_direct - S3_4_diag).max())

# Đo thời gian thực thi (microsecond)
t_direct = timeit.timeit(lambda: S3 @ S3 @ S3 @ S3, number=10000) / 10000 * 1e6
t_diag   = timeit.timeit(lambda: P3 @ np.diag(w3**4) @ P3_inv, number=10000) / 10000 * 1e6

print(f"Thời gian nhân trực tiếp: {t_direct:.3f} us")
print(f"Thời gian qua PDP^-1 (đã có P, D): {t_diag:.3f} us")

### B6. Phân tích SVD, Biểu đồ giá trị suy biến & Định lý Eckart-Young

In [ ]:
# === NHIỆM VỤ B6 ===
U, s_svd, Vt = np.linalg.svd(X, full_matrices=False)

# Vẽ biểu đồ giá trị suy biến (Scree plot)
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(s_svd) + 1), s_svd, 'ro-', linewidth=2, markersize=8)
plt.title("Biểu đồ giá trị suy biến (Scree Plot of Singular Values)")
plt.xlabel("Chỉ số giá trị suy biến (k)")
plt.ylabel("Giá trị suy biến (sigma_k)")
plt.xticks(range(1, len(s_svd) + 1))
plt.show()

print("=== B6: Xấp xỉ hạng k tốt nhất và Sai số Eckart-Young ===")
for k in [1, 2, 3]:
    X_hat_k = U[:, :k] @ np.diag(s_svd[:k]) @ Vt[:k, :]
    err_actual_2 = np.linalg.norm(X - X_hat_k, 2)
    err_theory_2 = s_svd[k]  # Định lý Eckart-Young
    err_actual_F = np.linalg.norm(X - X_hat_k, 'fro')
    err_theory_F = np.sqrt(np.sum(s_svd[k:]**2))
    
    print(f"Rank k = {k}:")
    print(f"  Spectral norm error ||X - X_hat||_2: Thực tế = {err_actual_2:.6f} | Lý thuyết (sigma_{k+1}) = {err_theory_2:.6f}")
    print(f"  Frobenius norm error ||X - X_hat||_F: Thực tế = {err_actual_F:.6f} | Lý thuyết = {err_theory_F:.6f}")

--- 
# NHIỆM VỤ C — CHƯƠNG 5: GRADIENT VÀ MA TRẬN HESSE

### Đặt bài toán hồi quy tuyến tính 2 đặc trưng:
- $x_{n1}$ = sepal_length (chuẩn hoá)
- $x_{n2}$ = sepal_width (chuẩn hoá)
- $y_n$ = petal_length (chuẩn hoá)
- Hàm mất mát: $L(w_1, w_2) = \frac{1}{N}\sum_{n=1}^{N}(w_1 x_{n1} + w_2 x_{n2} - y_n)^2$

In [ ]:
X1 = X[:, 0]
X2 = X[:, 1]
Y_target = X[:, 2]

def analytic_grad(w1, w2, x1, x2, y):
    n_samples = len(y)
    r = w1 * x1 + w2 * x2 - y
    dL_dw1 = (2.0 / n_samples) * np.sum(r * x1)
    dL_dw2 = (2.0 / n_samples) * np.sum(r * x2)
    return np.array([[dL_dw1, dL_dw2]])  # Dạng vectơ hàng 1x2

def analytic_hessian(x1, x2):
    n_samples = len(x1)
    H11 = (2.0 / n_samples) * np.sum(x1 ** 2)
    H22 = (2.0 / n_samples) * np.sum(x2 ** 2)
    H12 = (2.0 / n_samples) * np.sum(x1 * x2)
    return np.array([[H11, H12], [H12, H22]])

### C1 & C2. Gradient giải tích và Ma trận Hesse
Chứng minh Hesse độc lập với $w$ và kết luận tính lồi.

In [ ]:
# === NHIỆM VỤ C1 & C2 ===
g0 = analytic_grad(0.0, 0.0, X1, X2, Y_target)
H_mat = analytic_hessian(X1, X2)
eigvals_H = np.linalg.eigvalsh(H_mat)

print("=== C1: Gradient giải tích tại w=(0,0) (Dạng 1x2) ===")
print(g0, "| Shape:", g0.shape)

print("\n=== C2: Ma trận Hesse (không phụ thuộc w) ===")
print(H_mat)
print("Giá trị riêng của Hesse:", eigvals_H)
print("Tất cả giá trị riêng > 0?", np.all(eigvals_H > 0))
print("-> Kết luận: L(w1, w2) là hàm lồi nghiêm ngặt (strictly convex).")

### C3. Đồ thị tính toán (Computational Graph) cho N=2

In [ ]:
# === NHIỆM VỤ C3 ===
x11, x12, y1 = X1[0], X2[0], Y_target[0]
x21, x22, y2 = X1[1], X2[1], Y_target[1]

w1_val, w2_val = 0.5, -0.3

# Chạy lan truyền tiến (Forward)
z1 = w1_val * x11 + w2_val * x12
r1 = z1 - y1
s1 = r1**2

z2 = w1_val * x21 + w2_val * x22
r2 = z2 - y2
s2 = r2**2
L_val = 0.5 * (s1 + s2)

# Lan truyền ngược (Backpropagation) tính dL/dw1
dL_ds1 = 0.5
dL_dr1 = dL_ds1 * (2 * r1)
dL_dz1 = dL_dr1 * 1.0
dL_dw1_m1 = dL_dz1 * x11

dL_ds2 = 0.5
dL_dr2 = dL_ds2 * (2 * r2)
dL_dz2 = dL_dr2 * 1.0
dL_dw1_m2 = dL_dz2 * x21

dL_dw1_backprop = dL_dw1_m1 + dL_dw1_m2
dL_dw1_analytic = analytic_grad(w1_val, w2_val, X1[:2], X2[:2], Y_target[:2])[0, 0]

print(f"=== C3: Đối chiếu Backpropagation N=2 ===")
print(f"dL/dw1 (tính qua đồ thị tính toán): {dL_dw1_backprop:.6f}")
print(f"dL/dw1 (công thức C1): {dL_dw1_analytic:.6f}")
print("Khớp hoàn toàn?", np.isclose(dL_dw1_backprop, dL_dw1_analytic))

### C4. Kiểm chứng bằng Vi phân tự động (PyTorch autograd)
Kiểm tra tại 3 điểm $w$ khác nhau và báo cáo sai số tuyệt đối.

In [ ]:
# === NHIỆM VỤ C4 ===
X1_t = torch.tensor(X1, dtype=torch.float64)
X2_t = torch.tensor(X2, dtype=torch.float64)
Y_t  = torch.tensor(Y_target, dtype=torch.float64)

def loss_torch(w):
    r = w[0] * X1_t + w[1] * X2_t - Y_t
    return torch.mean(r**2)

test_points = [(0.0, 0.0), (0.5, -0.3), (-1.2, 0.8)]

print("=== C4: So sánh Gradient Giải tích vs PyTorch Autograd ===")
print(f"{'Điểm w':^18} | {'Grad Giải tích (1x2)':^28} | {'Grad Autograd (1x2)':^28} | {'Sai số tuyệt đối':^15}")
print("-" * 98)

for w1_i, w2_i in test_points:
    g_ana = analytic_grad(w1_i, w2_i, X1, X2, Y_target)
    
    w_tensor = torch.tensor([w1_i, w2_i], dtype=torch.float64, requires_grad=True)
    L_t = loss_torch(w_tensor)
    L_t.backward()
    
    # Autograd trả về dạng cột (2,), chuyển vị thành hàng (1, 2)
    g_torch = w_tensor.grad.detach().numpy().reshape(1, 2)
    abs_err = np.max(np.abs(g_ana - g_torch))
    
    print(f"({w1_i:5.2f}, {w2_i:5.2f}) | {np.array2string(g_ana, precision=6):^28} | {np.array2string(g_torch, precision=6):^28} | {abs_err:.2e}")

### C5. Khai triển Taylor bậc hai quanh điểm $w=(0,0)$

In [ ]:
# === NHIỆM VỤ C5 ===
def L_func(w1, w2):
    return np.mean((w1 * X1 + w2 * X2 - Y_target)**2)

L0 = L_func(0.0, 0.0)
g0_vec = analytic_grad(0.0, 0.0, X1, X2, Y_target).flatten()
H_m = analytic_hessian(X1, X2)

print("=== C5: Kiểm tra Khai triển Taylor Bậc hai ===")
print(f"L(0,0) = {L0:.6f}")
print(f"Grad L(0,0) = {g0_vec}")
print(f"Hesse = {H_m}")

print("\nSo sánh giá trị L thực tế và giá trị Taylor bậc 2:")
for w1_i, w2_i in [(0.3, 0.4), (-0.7, 1.1), (1.5, -0.2)]:
    w_v = np.array([w1_i, w2_i])
    L_real = L_func(w1_i, w2_i)
    L_taylor = L0 + g0_vec @ w_v + 0.5 * (w_v @ H_m @ w_v)
    diff = abs(L_real - L_taylor)
    print(f"w=({w1_i:4.1f}, {w2_i:4.1f}): L_real = {L_real:.8f} | L_taylor = {L_taylor:.8f} | Sai số = {diff:.2e}")